In [6]:
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv

In [7]:
load_dotenv()

True

In [8]:
with open('pg_essays/all_essays.json','r',encoding='utf-8') as f:
    essays = json.load(f)

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap=150,
    length_function=len,
    is_separator_regex=False
)

In [10]:
document=[]

for essay in essays:
    chunks = text_splitter.split_text(essay['text'])

    for chunk in chunks:
        document.append({
            'page_content':chunk,
            'metadata':{
                'title':essay['title'],
                'url':essay['url'],
                "date": essay['date']
            }
        })

In [11]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = Chroma.from_texts(
    texts=[doc["page_content"] for doc in document],
    metadatas=[doc["metadata"] for doc in document],
    embedding=embeddings,
    persist_directory="./pg_chroma_db"
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2603.10it/s]


In [12]:
retriever = vector_db.as_retriever(search_type='similarity',search_kwargs={'k':4})

In [13]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001D12EE7B320>, search_kwargs={'k': 4})

In [14]:
retriever.invoke("How to think of ideas?")

[Document(id='52977bea-20b4-47bf-9619-ac475c51428e', metadata={'url': 'https://paulgraham.com/top.html', 'title': 'The Top Idea in Your Mind', 'date': 'July 2010'}, page_content="The Top Idea in Your Mind\nWant to start a startup?\nGet funded by\nY Combinator\n.\nJuly 2010\nI realized recently that what one thinks about in the shower in the\nmorning is more important than I'd thought.  I knew it was a good\ntime to have ideas.  Now I'd go further: now I'd say it's hard to\ndo a really good job on anything you don't think about in the shower.\nEveryone who's worked on difficult problems is probably familiar\nwith the phenomenon of working hard to figure something out, failing,\nand then suddenly seeing the answer a bit later while doing something\nelse. There's a kind of thinking you do without trying to.  I'm\nincreasingly convinced this type of thinking is not merely helpful\nin solving hard problems, but necessary.  The tricky part is, you\ncan only control it indirectly.\n[\n1\n]\nI